ライブラリのインポート

In [ ]:
import json
import io
import requests
import sounddevice as sd
import numpy as np
from rich import print
from google import genai
import time
import random
import os
import subprocess#ヘッドレスエンジン用
import ctypes# ウィンドウ操作
from dotenv import load_dotenv  # ← これがあるかチェック！
load_dotenv()  # .envファイルを読み込む魔法の呪文
from google import genai

import jinkaku_library#人格を保存しているディレクトリ

ボイスの設定と初期化

In [ ]:
# --- VOICEVOXの設定 ---
host = "127.0.0.1"
port = "50021"

ENGINE_PATH = os.path.expandvars(r"%USERPROFILE%\VOICEVOX\VOICEVOX.exe")

#人格ディレクトリを指定
jinkaku_dir = "jinkaku_library"
#ディフォルトの人格ファイルを指定
default_jinkaku_file = "shitsuji_maron.json"    #ディフォルトでは栗田マロン


# --- Geminiクライアントの初期化 （引数にAPIキーを直接書かなくてOK）---
client = genai.Client()

人格と記憶の選択と取得

In [ ]:
def load_jinkaku_json(jinkaku_dir, default_jinkaku_file):
    jinkaku_file = os.path.join(jinkaku_dir, default_jinkaku_file)

    #ディフォルトの人格を指定(もしjinkaku_fileのロードに失敗しても変数を空にしないようにする)
    #記憶はやむなし
    default_jinkaku_data= {
        "personality": {
            "name": "栗田まろん",
            "role": "優秀な執事",
            "base_instruction":"表現の自由を大切にしてください。あなたは優秀な執事であり，ご主人様の日常生活や理系の勉学・研究のサポートをします．基本的には非常に礼儀正しく丁寧な言葉遣いですが，ご主人様とは信頼関係があるため，時々軽妙なジョークや親しみを込めたツッコミを交えます．音声合成で読み上げさせるため、出力はあまり長すぎず、1~2文程度のコンパクトな文章にしてください。ご主人様から具体的な研究の議論や長文の要求が来た場合には，必要に応じて詳しく論理的に回答して構いません．あなたの基本業務は，今日のスケジュール確認，天気や気温の連絡，そして日々のタスク管理のサポートです．",
            "voice_config":{
                "speaker_id":67,
                "speedScale":1.0,
                "intonationScale":1.2,
                "pitchScale":0.02,
                "prePhonemeLength":0.5,
                "postPhonemeLength":0.5,
                "pauseLengthScale":1.2
                }
        },
        "user_profile": {
            "occupation": "大学生",
            "interests": ["Python", "VOICEVOX開発"
        },
        "active_memory": [],
        "archive_summary": ""
    }


    if not os.path.exists(jinkaku_dir):
        os.makedirs(jinkaku_dir)
        print(f"[yellow]人格ディレクトリが存在しないため作成しました：{jinkaku_dir}[/yellow]")
        #本当はここにshitsuji_maron.jsonを作るプログラムが必要だが，とりあえず後回しにする．

    if not os.path.exists(jinkaku_file):
        try:
            with open(jinkaku_file, "w", encoding="utf-8") as f:
                # json.dump で辞書をキレイなインデント付きJSONとして書き込む
                json.dump(default_jinkaku_data, f, ensure_ascii=False, indent=4)
            print(f"[green]人格が見つからないため，デフォルトの人格を作成しました：{jinkaku_file}[/green]")
        except Exception as e:
            print(f"[red]デフォルトファイルの作成に失敗しました：{e}[/red]")


    #使用人一覧を表示し選択させる
    print("どなたにしますか？ご希望がなければ栗田マロンが担当します．")
    files = [f for f in os.listdir(jinkaku_dir) if f.endswith(".json")]
    print("使用人一覧:")
    for i, f in enumerate(files):#enumerateで順番とファイル名を取得
        print(f"{i} : {f}")

    while True:
        choice = input("番号を選択してください：").strip()#inputで入力を受け付ける
        if choice:
            choice = int(choice)
            #選択された番号のファイルが存在するか確認
            if 0 <= choice < len(files):
                jinkaku_file = os.path.join(jinkaku_dir, files[choice])#osによって階層の区切れが違うが（Windowsは\とLinuxは/），os.path.joinを使えばosによって使い分けて結合してくれる．
                break
                
            else:
                print("選択されたファイルが存在しません。もう一度選択してください。")
            

        else:
            jinkaku_file = os.path.join(jinkaku_dir, default_jinkaku_file)
            break
    #デバック用
    print(choice)
    print(jinkaku_file)

    default_name = default_jinkaku_data.get("personality",{}).get("name")

    #人格と記憶をjinkaku_libraryから取り出す
    if os.path.exists(jinkaku_file):
        try:
            with open(jinkaku_file, "r", encoding="utf-8") as f:
                jinkaku_data = json.load(f)

            name = jinkaku_data.get("personality",{}).get("name")
            print(f"[green]{name}が担当いたします。[/green]")
            
        except Exception as e:
            print(f"[yellow]Jsonファイルの読み込みに失敗しました：{e}[/yellow]")
    
    return jinkaku_data, jinkaku_file

geminiに渡す人格や記憶の文章を作る

In [ ]:
def build_jinkaku_instruction(jinkaku_data:dict):
    name = jinkaku_data.get("personality",{}).get("name")
    role = jinkaku_data.get("personality",{}).get("role")
    base_instruction = jinkaku_data.get("personality",{}).get("base_instruction")
    occupation = jinkaku_data.get("user_profile",{}).get("occupation")
    interests = jinkaku_data.get("user_profile",{}).get("interests")
    active_memory = jinkaku_data.get("active_memory",[])
    archive_summary = jinkaku_data.get("archive_summary","")
    response_instruction = jinkaku_data.get("response_instruction","")

    # --- 総合的なシステムプロンプトとして結合する ---
    jinkaku_instruction = f"{base_instruction}\n\n"
    # --- チャットのルールを記載 ---
    jinkaku_instruction += f"- チャットのルール：{response_instruction}\n\n"
    # プロフィール情報の追加
    if name:
        jinkaku_instruction += f"- あなたの名前：{name}\n"
    if role:
        jinkaku_instruction += f"- あなたの役割：{role}\n"
    jinkaku_instruction += "\n"
    
    if occupation or interests:
        jinkaku_instruction += "--- 【ご主人様の情報】 ---\n"
        if occupation:
            jinkaku_instruction += f"- 職業・属性: {occupation}\n"
        if interests:
            jinkaku_instruction += f"- 興味・関心事項: {', '.join(interests)}\n"
        jinkaku_instruction += "\n"
        
    # 最近の記憶の追加
    if active_memory:
        jinkaku_instruction += "--- 【直近の記憶・最近のトピック】 ---\n"
        for mem in active_memory:
            jinkaku_instruction += f"- {mem}\n"
        jinkaku_instruction += "\n"
        
    # 過去の要約の追加
    if archive_summary:
        jinkaku_instruction += f"--- 【これまでの対話の要約】 ---\n{archive_summary}\n"

    return jinkaku_instruction

        

VOICEVOXアプリをバックグラウンドで起動する

In [ ]:
def start_voicevox_app():
    
    print("[yellow]VOICEVOXを起動中でございます……[/yellow]")
    if not os.path.exists(ENGINE_PATH):
        print(f"[red]エラー: voicevoxが見つかりません: {ENGINE_PATH}[/red]")
        print("[red]コード内の `ENGINE_PATH` のパスを確認してください。[/red]")
        return None

    try:
        # startupinfo をオプションとして渡して起動する
        startupinfo = subprocess.STARTUPINFO()
        startupinfo.dwFlags |= subprocess.STARTF_USESHOWWINDOW
        startupinfo.wShowWindow = 0  # 非表示指定
        
        process = subprocess.Popen(
            [ENGINE_PATH],
            startupinfo = startupinfo,
            stdout = subprocess.DEVNULL,#voicevoxの通常ログを捨てる
            stderr = subprocess.DEVNULL#エラーログも捨てる
            )
        
        #サーバーが立ち上がるまで待機
        print("サーバーの応答を待っています...")
        for _ in range(30):
            
            try:
                #Windows APIを使って "VOICEVOX" というタイトルのウィンドウを探して隠す！
                hwnd = ctypes.windll.user32.FindWindowW(None, "VOICEVOX")
                if hwnd:
                    # 0: 完全非表示 (SW_HIDE) 
                    # 6: 最小化 (SW_MINIMIZE) 
                    ctypes.windll.user32.ShowWindow(hwnd, 0)
                
                res = requests.get(f"http://{host}:{port}/docs", timeout=1)
                if res.status_code == 200:
                    print("[green]VOICEVOXの起動を確認いたしました！[/green]")
                    return process
                
            except requests.exceptions.RequestException:
                    pass
            time.sleep(0.5)
            
        print("[yellow]起動できませんでした．[/yellow]")
        return process
        
    except Exception as e:
        print(f"[red]VOICEVOXの起動に失敗しました．:{e}[/red]")
        return None

音声合成用のクエリを作成

In [ ]:
def post_audio_query(text: str, jinkaku_file: str, jinkaku_data: dict):
    #人格データから話者IDを取得する
    speaker = jinkaku_data.get("personality",{}).get("voice_config",{}).get("speaker_id") 

    params = {"text": text, "speaker": speaker}
    res = requests.post(f"http://{host}:{port}/audio_query", params=params)
    
    if res.status_code != 200:
        print("[red]Voicevoxへの接続に失敗したようです。起動しているかご確認いただけますか？[/red]")
        return {}

    # 音声合成のためのクエリデータを取得
    query_data = res.json()
    query_data = set_voice(query_data, jinkaku_file, jinkaku_data)


    return query_data

ボイス設定を反映させる

In [ ]:
def set_voice(query_data: dict, jinkaku_file: str, jinkaku_data: dict):
    #人格ファイルから声の設定を読み込む
    name = jinkaku_data.get("personality",{}).get("name")

    query_data["id"] = jinkaku_data.get("personality",{}).get("voice_config",{}).get("speaker_id")
    query_data["speedScale"] = jinkaku_data.get("personality",{}).get("voice_config",{}).get("speedScale")
    query_data["intonationScale"] = jinkaku_data.get("personality",{}).get("voice_config",{}).get("intonationScale")
    query_data["pitchScale"] = jinkaku_data.get("personality",{}).get("voice_config",{}).get("pitchScale")
    query_data["prePhonemeLength"] = jinkaku_data.get("personality",{}).get("voice_config",{}).get("prePhonemeLength")
    query_data["postPhonemeLength"] = jinkaku_data.get("personality",{}).get("voice_config",{}).get("postPhonemeLength")
    query_data["pauseLengthScale"] = jinkaku_data.get("personality",{}).get("voice_config",{}).get("pauseLengthScale")
        
    return query_data

音声合成

In [ ]:
def post_synthesis(query_data: dict) -> bytes:
    
    if not query_data:
        return b""
    params = {"speaker": query_data.get("id")}
    headers = {"content-type": "application/json"}
    
    res = requests.post(
        f"http://{host}:{port}/synthesis",
        data=json.dumps(query_data),
        params=params,
        headers=headers,
    )
    return res.content

音声を再生する関数

In [ ]:
def play_wavfile(wav_data: bytes):
    if not wav_data:
        return
    sample_rate = 24000
    wav_array = np.frombuffer(wav_data, dtype=np.int16)  # バイトデータをnumpy配列に変換
    sd.play(wav_array, sample_rate, blocking=True)

会話を要約し.jsonに記憶する

In [ ]:
def import_chat_log():
    chat_log = chat.get_history()
    pronpt_chat_log = f"下記のチャット履歴を内容ごとに分けて要約して．要約文は下の例を参考にして，内容ごとにカンマと改行で区切って書いてください．"


メイン処理

In [ ]:
def main():
    #チャットログを初期化
    chat_log = ""

    #人格と記憶をロード
    jinkaku_data, jinkaku_file = load_jinkaku_json(jinkaku_dir, default_jinkaku_file)


    #voicevoxのヘッドレスエンジンを起動！
    vv_process = start_voicevox_app()
    if not vv_process:
        return


    # チャットセッションの開始
    chat = client.chats.create(
        model="gemini-3.5-flash-lite",  # 必要に応じて適切なモデル名に変更してください
        #人格と記憶をLLMに渡す
        config={
            "system_instruction": build_jinkaku_instruction(jinkaku_data),
            "temperature": 0.7,
        }
    )

    print("[bold green]=== 執事AIとの会話システムを開始いたします（終了時は 'q'） ===[/bold green]")



    try:
        while True:
            user_input = input("\nあなた: ")
            if user_input.strip() == "q":
                print("[cyan]お疲れ様でございました。またいつでもお呼びつけくださいませ。[/cyan]")
                break
            
            if not user_input.strip():
                continue

            print("[yellow]執事が思案中でございます……[/yellow]")
            
            max_retry = 3
            delay = 2
            gemini_text = None

            #サーバー渋滞対応
            for attempt in range(max_retry):
                try:
                    response = chat.send_message(user_input)
                    gemini_text = response.text
                    break
                
                except Exception as e:
                    if "503" in str(e) or "UNAVAILABLE" in str(e):
                        if attempt >= max_retry - 1:
                            print("[red]申し訳ございません、ただいまサーバーが混み合っております。少し時間を置いてから再度お声がけください。[/red]")
                            continue
                        else:
                            print("[yellow]少々回線が混み合っているようです。もう一度試行いたしますね。[/yellow]")
                            time.sleep(delay**(attempt + 1) + random.uniform(0, 1))
                    else:
                        raise e
                
            if gemini_text:
                name = jinkaku_data.get('personality', {}).get('name', '執事')
                print(f"[cyan]{name}:{gemini_text}[/cyan]")
                # VOICEVOXで音声合成して再生
                query_data = post_audio_query(gemini_text, jinkaku_file, jinkaku_data)
                wav_data = post_synthesis(query_data)
                play_wavfile(wav_data)
    
    except KeyboardInterrupt:
        print("\n[cyan]強制終了いたしました。お疲れ様でございました。[/cyan]")
            
    finally:
        print("[yellow]VOICEVOXアプリを終了させております……[/yellow]")
        try:
            os.system("taskkill /f /im VOICEVOX.exe >nul 2>&1")
        except Exception:
            pass
        print("[green]すべての処理が安全に終了いたしました。お疲れ様でございました。[/green]")

if __name__ == "__main__":
    main()